In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv

In [2]:
# load the environment variable
load_dotenv()

True

In [3]:
# Define the Hugging Face endpoint
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    # repo_id="openai/gpt-oss-120b",
    task="text-generation",
    max_new_tokens=2048,
)

# Define the model
model = ChatHuggingFace(llm=llm)

e:\Langgraph CampusX\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class LLMState(TypedDict):

    question: str
    answer: str

In [8]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from the state
    question = state['question']

    # define the prompt
    prompt = f"Answer the following question: {question}"

    # call the model
    answer = model.invoke(prompt)

    # update the state with the answer
    state["answer"] = answer.content

    return state

In [9]:
# define the graph
graph = StateGraph(LLMState)

# add nodes to the graph
graph.add_node("llm_qa", llm_qa)

# add edges tp the graph
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

workflow = graph.compile()

In [10]:
# define the initial state
initial_state = {'question': "What is the difference between SQL Truncate, delete and drop?"}

final_state = workflow.invoke(initial_state)

print(final_state["answer"])

Certainly! The SQL commands `TRUNCATE`, `DELETE`, and `DROP` are used to manage data and structures in a database, but they serve different purposes and have different characteristics:

1. **TRUNCATE**:
   - **Purpose**: `TRUNCATE` is used to remove all rows from a table. It is faster than `DELETE` because it does not log each deleted row, but it does log the truncation of the table itself.
   - **Ownership**: `TRUNCATE` is faster and removes all rows from the table, but it does not log individual row deletions. It can only be used with tables, not views.
   - **Constraints**: If the table has foreign key constraints, `TRUNCATE` will fail if there are any rows in the dependent tables. If there are no such constraints, `TRUNCATE` still marks the table as having been truncated.
   - **Example**: `TRUNCATE TABLE my_table;`

2. **DELETE**:
   - **Purpose**: `DELETE` is used to remove rows from a table where a specified condition is met. If no condition is specified, all rows are deleted, s